# 텍스트 특징 추출 (TF-IDF)

### 1. 샘플 코퍼스로 연습

In [1]:
sample_corpus = [
    '자연어처리 강의를 시작하겠습니다.',
    '자연어처리는 재미있습니다.',
    '밥을 먹고 강의를 듣고 있습니다.',
    '이번 자연어처리 강의는 한국어 자연어처리입니다.'
]

In [2]:
import sklearn

In [3]:
from konlpy.tag import Okt

def my_tokenizer(text):
    return Okt().nouns(text)

In [5]:
# tfidVectorizer 객체 생성
#from sklearn.feature_extraction.text import TfidfVectorizer
#vectorizer = TfidfVectorizer(tokenizer = my_tokenizer)

from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(tokenizer = my_tokenizer, token_pattern=None)

# 특징 집합과 관련 데이터 모델을 생성
vectorizer.fit(sample_corpus)
print(vectorizer.get_feature_names_out())
# 특징 벡터 추출(빈도 수 및 문서必)
sample_dtm = vectorizer.transform(sample_corpus)
type(sample_dtm)#   sparse array 타입
sample_dtm.toarray()#   일반 list형태로


['강의' '밥' '시작' '이번' '자연어' '처리' '한국어']


array([[1, 0, 1, 0, 1, 1, 0],
       [0, 0, 0, 0, 1, 1, 0],
       [1, 1, 0, 0, 0, 0, 0],
       [1, 0, 0, 1, 2, 2, 1]])

### 2. 다음 영화 리뷰로 연습

In [1]:
#   1) 데이터 불러오기(max() token 확인 후 분석 진행)
import pandas as pd
from konlpy.tag import Okt
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter

df_file_01 = r'.\data\daum_movie_review.csv'

movie_review = pd.read_csv(df_file_01, dtype='str')

In [2]:
#   df 구조 확인
movie_review.head()

,review,rating,date,title
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워
4,재미있다,10,2018.10.20,인피니티 워


In [3]:
#   2) 형태소 변환기 생성
tokenizer = Okt()

def my_tokenizer(text):
    return tokenizer.nouns(text)

In [4]:
#   3) 한글 제외 글자 제거:
#      movie_review["review"].str: '.str'활용하여 
#      "review" 열 입력 데이터에 대해일괄 문자형 전환
movie_review['set_review'] = movie_review["review"].str.replace("[^ㅏ-ㅣㄱ-ㅎ가-힣]+", " ", regex=True)

In [5]:
#   4) 개별 행에 대해 my_tokenizer 적용 후 'tokens' 열 생성
movie_review['tokens'] = movie_review['set_review'].apply(my_tokenizer)

In [6]:
#   5) 개별 행(list)에 대해 문자열 추출
#       ① movie_review['tokens']에서 tokens(리스트) 추출
#       ② 개별 word(단어) 추출
all_token = [word for tokens in movie_review['tokens'] for word in tokens]
count_tokens = Counter(all_token)

print(len(count_tokens))

10394


In [ ]:
#   6) 불용어 설정
stop_tokens = []

#   7) dtm(문서-단어 행렬) 생성 위한 CountVectorizer 모델 설정
vectorizer = CountVectorizer(tokenizer = my_tokenizer,
                             token_pattern=None,
                             stop_words = stop_tokens,
                             max_features = 1000,
                             max_df = 0.8
                             )

In [ ]:
# 8) 특징 집합과 관련 데이터 모델을 생성

#   'set_review' 기반 단어사전 구축
vectorizer.fit(movie_review['set_review'])
print(vectorizer.get_feature_names_out())

#   단어사전 기반 dtm 변환
sample_dtm = vectorizer.transform(movie_review['set_review'])
print(sample_dtm.toarray())

TypeError: fit() got an unexpected keyword argument 'token_pattern'